In [59]:
import scripts.init_gpu as init_gpu
import scripts.init_dataset as init_dataset
import pandas as pd
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '1'
init_gpu.initialize_gpus()

locations = ['LOC2', 'LOC3']

print("Loading Dataset...")
# load the dataset
df = pd.read_csv(
    f"../dataset/processed/{locations[0]}-{locations[1]}-Date-ID.csv")

df.drop(['Date', 'ID'], inplace=True, axis=1)
df = df.iloc[:, :66]


length = len(df.columns) - 2  # subtract the two label columns

# get train-test set
train_df, test_df, train_web_sam1ples, test_web_samples = init_dataset.get_sample(
    df, locations, range(1500), 1200)

train_df.head()

Num GPUs Available:  0
Loading Dataset...
Training Websites: [1309, 228, 51, 563, 501, 457, 285, 209, 1385, 1116, 178, 1209, 864, 65, 61, 191, 447, 476, 1034, 1232, 54, 1149, 407, 1466, 1330, 1436, 1490, 859, 451, 919, 1206, 569, 13, 326, 1429, 865, 696, 1468, 318, 440, 689, 1492, 189, 778, 198, 735, 704, 1236, 541, 88, 940, 1098, 255, 775, 161, 1130, 600, 1287, 1266, 740, 1182, 393, 142, 93, 1354, 466, 592, 163, 1482, 206, 1456, 1462, 928, 1301, 747, 333, 758, 727, 429, 1372, 546, 1399, 1327, 146, 1247, 1300, 350, 1093, 1495, 334, 946, 777, 552, 1310, 1140, 449, 1402, 664, 114, 469, 1486, 646, 821, 548, 135, 432, 1161, 644, 435, 1342, 1022, 810, 1316, 939, 292, 542, 1493, 505, 1478, 1103, 538, 1197, 877, 1195, 817, 741, 1404, 283, 1043, 1010, 186, 96, 224, 313, 1285, 327, 1487, 1221, 130, 788, 781, 1220, 958, 1083, 514, 1133, 23, 234, 1099, 1419, 1312, 1463, 1498, 601, 890, 323, 929, 6, 539, 1025, 365, 1039, 217, 1280, 611, 1308, 1338, 1415, 1477, 1366, 765, 330, 1104, 1086, 1, 1226, 

c:\Users\kaush\Documents\Python Projects\DoH-Synthesis\code\scripts\init_dataset.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df.sort_values(by=["Location"], inplace=True)


,Location,Website,0,1,2,3,4,5,6,7,...,54,55,56,57,58,59,60,61,62,63
0,LOC2,0,88,-64,94,62,33,33,-33,-268,...,40,64,33,43,65,33,-138,-33,-139,-33
1,LOC2,987,-1,-40,-64,88,94,69,33,33,...,-185,-33,43,67,33,-170,-33,43,78,33
2,LOC2,987,-1,-40,-64,88,94,69,33,33,...,-177,-33,43,67,33,-170,-33,43,78,33
3,LOC2,987,88,-64,94,69,33,33,-33,-280,...,43,67,33,-170,-33,43,78,33,-176,-33
4,LOC2,987,88,94,69,33,-64,33,-33,-276,...,43,78,33,-177,-33,43,67,33,-171,-33


In [60]:
import numpy as np
import pandas as pd

def get_random_location_pairs(df, source_location='LOC1', target_location='LOC2', num_pairs=50, website_col='Website', location_col='Location', random_state=None):
    """
    Returns 50 random (LOC1, LOC2) sample pairs from 50 randomly selected websites.
    Each pair is (sample_from_LOC1, sample_from_LOC2) for the same website.
    """
    # Find websites present in both locations
    websites_loc1 = set(df[df[location_col] == source_location][website_col])
    websites_loc2 = set(df[df[location_col] == target_location][website_col])
    common_websites = list(websites_loc1 & websites_loc2)
    if len(common_websites) < num_pairs:
        raise ValueError(f"Not enough common websites to sample {num_pairs} pairs.")

    # Randomly select websites
    selected_websites = np.random.choice(common_websites, size=num_pairs, replace=False)

    pairs = []
    for web_id in selected_websites:
        # Random sample from LOC1 for this website
        loc1_samples = df[(df[website_col] == web_id) & (df[location_col] == source_location)]
        loc2_samples = df[(df[website_col] == web_id) & (df[location_col] == target_location)]
        sample1 = loc1_samples.sample(n=1, random_state=random_state)
        sample2 = loc2_samples.sample(n=1, random_state=random_state)
        pairs.append((sample1.iloc[0, 2:].values, sample2.iloc[0, 2:].values))

    return pairs

# Example usage:
pairs = get_random_location_pairs(train_df, source_location='LOC2', target_location='LOC3', num_pairs=50)


In [ ]:
import numpy as np

def mse_to_target_location(df, values, website_id, target_location, website_col='Website', location_col='Location'):
    """
    Given a feature array (values) and a website_id, select a random sample from the target_location
    with the same website_id and compute the MSE between the values and the target sample's features.
    
    Args:
        df (pd.DataFrame): The dataframe containing the data.
        values (np.ndarray): Feature array to compare (should match df.iloc[:, 2:].shape[1]).
        website_id: The website ID to match.
        target_location: The location to select the target sample from.
        website_col: Name of the website column.
        location_col: Name of the location column.
        
    Returns:
        mse (float): Mean squared error between values and the selected target sample.
        target_sample (np.ndarray): The selected target sample's feature array.
    """
    # Filter for target location and website_id
    candidates = df[(df[website_col] == website_id) & (df[location_col] == target_location)]
    if len(candidates) == 0:
        raise ValueError(f"No sample found for website_id={website_id} at location={target_location}")
    
    # get mean of the samples
    target_values = candidates[:, 2:].median(axis=0).to_numpy()
        
    # Compute MSE
    mse = np.mean((values - target_values) ** 2, axis=1)
    return mse

# Example usage:
web_id = 12
source_location = 'LOC2'
trace = train_df[(train_df['Website'] == web_id) & (train_df['Location'] == source_location)].iloc[0, 2:].values
mse = mse_to_target_location(train_df, trace, web_id, target_location='LOC3')
print(f"MSE to target location: {mse}")

MSE to target location: 4285.125


In [261]:
source_test_df = train_df[train_df['Location'] == 'LOC2'].sample(n=5)
source_websites = source_test_df.Website.values
source_test_traces = "\n".join(
    [f"{trace}" for trace in source_test_df.iloc[:, 2:].values.tolist()]
)

In [ ]:
init_prompt="""You are given a packent counts of a DoH trace from a source location, Your task is to generate a DoH trace for the target location based on the source location's trace.
Positive values indicate uploads, while negative values indicate downloads. Each trace contains 64 values.

Here are some actual pairs of traces from the source and target locations. 
{sample_pairs}

Let's slowly change the following traces to the target location, by modifying the source trace slightly.  
{source_test_traces}


You should give the new traces in python in the format synthesized: List[List[int]]. Do not write or execute any code.
synthesized = 
"""

In [270]:
sample_pairs = "\n".join(
    [f"({pair[0]}, {pair[1]})" for pair in get_random_location_pairs(train_df, source_location='LOC2', target_location='LOC3', num_pairs=5)])

In [ ]:
print(init_prompt.format(sample_pairs=sample_pairs, source_test_traces=source_test_traces))

You are given a packent counts of a DoH trace from a source location, Your task is to generate a DoH trace for the target location based on the source location's trace.
Positive values indicate uploads, while negative values indicate downloads. Each trace contains 64 values.

Here are some actual pairs of traces from the source and target locations. 
([88 94 -64 61 33 33 -33 -273 -33 43 71 33 43 70 33 -163 -33 -167 -33 43 66
 33 43 65 33 43 75 33 -141 -33 -149 -33 -131 -33 43 73 33 40 75 33 -216
 -33 -175 -33 40 66 33 -159 -33 43 74 33 40 65 33 -136 -33 -220 -33 40 66
 33 -137 -33], [-64 88 94 61 33 33 -33 -268 -33 43 71 33 -270 -33 43 70 33 -164 -33 43 66
 33 43 65 33 -165 -33 -130 -33 43 75 33 -149 -33 43 73 33 40 75 33 -256
 -33 -193 -33 43 74 33 -219 -33 40 66 33 -37 -137 -33 40 65 33 -220 -33 43
 68 33 -141])
([-1 -32 -56 80 86 55 25 25 -25 -266 -25 35 59 25 -138 -25 35 61 25 -239
 -25 35 65 25 -243 -25 35 57 25 -370 -25 35 68 25 -165 -25 35 70 25 32 59
 25 -192 -25 -163 -25 32 57

In [ ]:
from API_KEY import GEMINI_API_KEY
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)
response = client.models.generate_content(
        model="gemma-3-27b-it", contents=init_prompt.format(sample_pairs=sample_pairs, source_test_traces=source_test_traces))

print(response.text)

```python
synthesized = [
    [-64, 88, 94, 62, 33, 33, -33, -290, -33, 43, 66, 33, -242, -33, 43, 63, 33, 43, 65, 33, -37, -356, -33, -255, -33, 43, 68, 33, -243, -33, 43, 64, 33, -159, -33, 43, 75, 33, -227, -33, 43, 69, 33, 43, 72, 33, -247, -33, -211, -33, 40, 68, 33, 40, 64, 33, 40, 63, 33, -205, -33, -188, -33, 43],
    [-1, -40, -64, 88, 94, 71, 33, 33, -33, -282, -33, 40, 71, 33, -204, -33, 43, 76, 33, 43, 75, 33, -216, -33, -141, -33, 43, 68, 33, -179, -33, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [-1, -40, -64, 88, 94, 63, 33, 33, -33, -269, -33, 43, 67, 33, -141, -33, 43, 65, 33, -162, -33, 43, 66, 33, -140, -33, 40, 66, 33, -137, -33, 40, 66, 33, -184, -33, 43, 73, 33, -187, -33, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [-64, 88, 94, 66, 33, 33, -33, -325, -33, 43, 70, 33, -226, -33, 40, 66, 33, 43, 73, 33, -215, -33, -214, -33, 43, 74, 33, -216, -33, 43, 64, 33, -214, -33, 0, 0,

In [307]:

import re
import ast

def parse_synthesized_from_response(response_text):
    """
    Extracts the synthesized: List[List[int]] from a python code block in the response text.
    Returns the synthesized list.
    Handles cases where the code block is missing and only the assignment is present.
    """
    # Try to extract the python code block first
    code_block_match = re.search(r"```python(.*?)```", response_text, re.DOTALL)
    if code_block_match:
        code_block = code_block_match.group(1)
    else:
        # If no code block, use the whole response text
        code_block = response_text

    # Extract the synthesized assignment (greedy to last closing bracket)
    synth_match = re.search(r"synthesized\s*=\s*(\[[\s\S]*\])", code_block)
    if not synth_match:
        raise ValueError("No synthesized list found in response.")
    synth_str = synth_match.group(1)

    # Remove trailing commas before closing brackets
    synth_str = re.sub(r',\s*\]', ']', synth_str)
    synth_str = re.sub(r',\s*\n\s*\]', ']', synth_str)
    synth_str = synth_str.strip()

    # Ensure the brackets are balanced
    open_brackets = synth_str.count('[')
    close_brackets = synth_str.count(']')
    if open_brackets > close_brackets:
        synth_str += ']' * (open_brackets - close_brackets)
    elif close_brackets > open_brackets:
        synth_str = synth_str.rstrip(']') + ']' * open_brackets

    # Safely evaluate the list
    synthesized = ast.literal_eval(synth_str)
    return synthesized


synthesized = parse_synthesized_from_response(response.text)

errors = []
for trace, website_id in zip(synthesized, source_websites):
    generated_mse = mse_to_target_location(train_df, trace, website_id, target_location='LOC3')
    errors.append(generated_mse)

In [294]:
sample_pairs = "\n".join(
    [f"({pair[0]}, {pair[1]})" for pair in get_random_location_pairs(train_df, source_location='LOC2', target_location='LOC3', num_pairs=5)])

current_synthesis_with_errors = ""
for original_source, synthesized_trace, error in zip(source_test_df.iloc[:, 2:].values.tolist(), synthesized, errors):
    current_synthesis_with_errors += f"({original_source}, {synthesized_trace}), Error: {error}\n"

In [304]:
optimizer_prompt="""You are given a packent counts of a DoH trace from a source location, Your task is to generate a DoH trace for the target location based on the source location's trace.
Positive values indicate uploads, while negative values indicate downloads. Each trace contains 64 values.

Here are some actual pairs of traces from the source and target locations. 
{sample_pairs}

Here are the traces you generated, along with their errors to the target location, lower errors indicate better traces:
(Original Source Trace, Synthesized Trace), Error: <error>
{current_synthesis_with_errors}


Change the traces slightly to reduce the errors, while still keeping the traces valid DoH traces.  
You should give the new traces in python block in the format synthesized: List[List[int]]. Do not write or execute any code.
synthesized = 
"""

In [305]:
optimizer_prompt = optimizer_prompt.format(sample_pairs=sample_pairs, current_synthesis_with_errors=current_synthesis_with_errors)
print(optimizer_prompt)

You are given a packent counts of a DoH trace from a source location, Your task is to generate a DoH trace for the target location based on the source location's trace.
Positive values indicate uploads, while negative values indicate downloads. Each trace contains 64 values.

Here are some actual pairs of traces from the source and target locations. 
([-1 -32 -56 80 86 54 25 25 -25 -261 -25 35 58 25 -146 -25 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0], [88 -64 94 62 33 33 -33 -269 -33 43 66 33 -154 -33 43 72 33 40 72 33 -229
 -33 -193 -33 43 67 33 40 67 33 -191 -33 -188 -33 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0])
([-64 88 94 73 33 33 -33 -285 -33 43 77 33 -142 -33 43 71 33 43 74 33 43 66
 33 43 70 33 -207 -33 -181 -33 -306 -33 -229 -33 43 75 33 43 68 33 -169
 -33 -193 -33 40 66 33 -232 -33 40 71 33 -168 -33 43 64 33 -155 -33 43 62
 33 -229 -33], [88 -64 94 73 33 33 -33 -280 -33 43 77 33 -165 -33 43 71 33 43 7

In [ ]:
response = client.models.generate_content(
        model="gemma-3-27b-it", contents=optimizer_prompt.format(sample_pairs=sample_pairs, current_synthesis_with_errors=current_synthesis_with_errors))

synthesized = parse_synthesized_from_response(response.text)
synthesized

[[-64,
  88,
  94,
  62,
  33,
  33,
  -33,
  -280,
  -33,
  43,
  66,
  33,
  -232,
  -33,
  43,
  63,
  33,
  43,
  65,
  33,
  -37,
  -346,
  -33,
  -245,
  -33,
  43,
  68,
  33,
  -233,
  -33,
  43,
  64,
  33,
  -159,
  -33,
  43,
  75,
  33,
  -217,
  -33,
  43,
  69,
  33,
  43,
  72,
  33,
  -237,
  -33,
  -201,
  -33,
  40,
  68,
  33,
  40,
  64,
  33,
  40,
  63,
  33,
  -205,
  -33,
  -188,
  -33,
  43],
 [-1,
  -40,
  -64,
  88,
  94,
  71,
  33,
  33,
  -33,
  -272,
  -33,
  40,
  71,
  33,
  -194,
  -33,
  43,
  76,
  33,
  43,
  75,
  33,
  -206,
  -33,
  -131,
  -33,
  43,
  68,
  33,
  -169,
  -33,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [-1,
  -40,
  -64,
  88,
  94,
  63,
  33,
  33,
  -33,
  -269,
  -33,
  43,
  67,
  33,
  -141,
  -33,
  43,
  65,
  33,
  -152,
  -33,
  43,
  66,
  33,
  -130,
  -33,
  40,
  66,
  33,
  -127,
  -33,
  40,